# Section 1: Ingestion Pipeline for GovTech Assessment
This notebook implements the Section 1 ingestion pipeline using Python and SQLite.
It reads `data/enrolments.csv`, applies cleansing and type conversions, and loads the cleaned data into the existing `data/courses.db` database.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

from gt_assessment.ingest import ingest_enrolments, load_and_clean_enrolments, create_database, validate_enrolment_load


## Section: Define SQLite Schema and Connection
Use the helper package in `gt_assessment.ingest` to create the database and required tables.


In [2]:
db_path = Path('data/courses.db')
conn = create_database(db_path)
print(f'Using existing database at: {db_path.resolve()}')
conn.close()


Using existing database at: C:\projects\GT_Assessment\data\courses.db


## Section: Extract Source Data
Read the source CSV file into a DataFrame and inspect the first few rows.


In [3]:
source_csv = Path('data/enrolments.csv')
df_raw = pd.read_csv(source_csv)
df_raw.head(10)


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used
0,1,P001,Alice Tan,1,15/01/2024,500.0,200.0,300.0
1,2,P002,Bob Lee,1,2024-01-15,500.0,250.0,250.0
2,3,P007,Grace Koh,1,20 Jan 2024,500.0,250.0,250.0
3,4,P009,Irene Chua,1,2024-01-25,500.0,200.0,300.0
4,5,P003,Charlie Ng,2,01-02-2024,600.0,300.0,300.0
5,6,P001,Alice Tan,2,2024/02/01,600.0,200.0,400.0
6,7,P004,Diana Lim,1,10 Feb 2024,500.0,150.0,350.0
7,8,P007,Grace Koh,2,2024-02-15,600.0,300.0,300.0
8,9,P009,Irene Chua,2,20/02/2024,600.0,250.0,350.0
9,10,P002,Bob Lee,3,2024-03-05,750.0,400.0,350.0


## Section: Transform and Clean Data
Normalize the date field, convert numeric columns, and remove invalid rows before loading.


In [4]:
df_clean = load_and_clean_enrolments(source_csv)
print('Rows after cleaning:', len(df_clean))
df_clean.head(10)


Rows after cleaning: 34


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used
0,1,P001,Alice Tan,1,2024-01-15,500.0,200.0,300.0
1,2,P002,Bob Lee,1,2024-01-15,500.0,250.0,250.0
2,3,P007,Grace Koh,1,2024-01-20,500.0,250.0,250.0
3,4,P009,Irene Chua,1,2024-01-25,500.0,200.0,300.0
4,5,P003,Charlie Ng,2,2024-02-01,600.0,300.0,300.0
5,6,P001,Alice Tan,2,2024-02-01,600.0,200.0,400.0
6,7,P004,Diana Lim,1,2024-02-10,500.0,150.0,350.0
7,8,P007,Grace Koh,2,2024-02-15,600.0,300.0,300.0
8,9,P009,Irene Chua,2,2024-02-20,600.0,250.0,350.0
9,10,P002,Bob Lee,3,2024-03-05,750.0,400.0,350.0


## Section: Load Data into SQLite Tables
Write the cleaned enrollment data into the SQLite database table.


In [5]:
db_path = ingest_enrolments(db_path, source_csv)
print(f'Enrolments data loaded to: {db_path.resolve()}')


Enrolments data loaded to: C:\projects\GT_Assessment\data\courses.db


## Section: Validate Ingestion Results
Query the SQLite database to confirm row counts and review a sample of ingested rows.


In [7]:
conn = sqlite3.connect(db_path)
row_count = pd.read_sql_query('SELECT COUNT(*) AS row_count FROM enrollments', conn)
preview = pd.read_sql_query('SELECT * FROM enrollments ORDER BY enrollment_id LIMIT 10', conn)
conn.close()
print(row_count)
preview


   row_count
0         34


,enrollment_id,participant_id,participant_name,course_id,course_date,amount,subsidy,credits_used
0,1,P001,Alice Tan,1,2024-01-15,500.0,200.0,300.0
1,2,P002,Bob Lee,1,2024-01-15,500.0,250.0,250.0
2,3,P007,Grace Koh,1,2024-01-20,500.0,250.0,250.0
3,4,P009,Irene Chua,1,2024-01-25,500.0,200.0,300.0
4,5,P003,Charlie Ng,2,2024-02-01,600.0,300.0,300.0
5,6,P001,Alice Tan,2,2024-02-01,600.0,200.0,400.0
6,7,P004,Diana Lim,1,2024-02-10,500.0,150.0,350.0
7,8,P007,Grace Koh,2,2024-02-15,600.0,300.0,300.0
8,9,P009,Irene Chua,2,2024-02-20,600.0,250.0,350.0
9,10,P002,Bob Lee,3,2024-03-05,750.0,400.0,350.0
